# Running SKIRT simulations

Before runnning a simulation, you should have SKIRT and PTS installed, and their dependencies. The SKIRT documentation has a tutorial: https://skirt.ugent.be/root/_installation_guide.html.

We create a Sim object by feeding it a Galaxy object as shown below. The Sim object requires a Galaxy object and path to your run directory, but we can vary additional parameters as well, such as:

max_radius : int or float
- The maximum radial distance (Kpc) of a star or gas particle. Particles outside of this radius are not used in the simualtion.
        If max_radius is None, all particles are kept.
        Default: None

import_velocity : bool
- Determines whether to include Doppler shift from velocity relative to model coordinates. Default: False. (NOTE: requires a "velocity" column in gas_cols and star_cols)

import_gas_temp : bool
- Determines whether to include gas temperature input files, which enables the option to set a maximum gas temperature in the simulation. Default: False. (NOTE: requires a "temperature" column in gas_cols)

n_random_projections : int
- The number of random projections generated by the simulation, in addition to the default xy, xz, and yz projections. For each random projection, SKIRT outputs a fits_file and sed_file.

n_photon_packets : int
- The number of photon packets launched in the simulation. Increasing the number photon packets makes the simulation more computationally expensive.

In [1]:

import sys
sys.path.append("../src")

import SKIRT_tools
import SKIRT_tools.skirt_tools as sktools

rundir = "/nfspool-0/home/jennydw/SKIRT-tools/tutorials/example" # Absolute path
# load galaxy from .hdf5 file
m12f = sktools.Galaxy("m12f_mass56000_cr_700", hdf5_file = "m12f_mass56000_cr_700.h5")
# input the simulation parameters you want
m12f_simulation = sktools.Sim(m12f, rundir, n_photon_packets=1e3)
# As a first test, we initialize the simulation with only 1000 photon packets.
# The more photon packets, the more computationally expensive; A rule of thumb is 50 photon packets per pixel.


No snapshot number found.


By default, the simulation will output three different projections: xy, yz, and zx, relative to the coordinates that are inputed into the Galaxy object. However, you can add as many instruments as you want. For example, if you wanted to add a random projection:

In [2]:

import SKIRT_tools.observation_utils as ou

inclination, azimuth, roll = ou.random_viewing_angles()
m12f_simulation.add_instrument("rand1", inclination, azimuth, roll)


build() writes the necessary data and parameter files SKIRT needs, and outputs them in the run directory. This only needs to be done once:

In [3]:

# creates a run directory and initializes parameter files
m12f_simulation.build()


All parameter and data files already in run directory.


The simulation can then be run with the following code. You may need to copy this script into a .py file and run it with the right environment and dependencies.

In [4]:

m12f_simulation.run(nthreads=4, nprocesses=2)


/nfspool-0/home/jennydw/SKIRT-tools/tutorials/example/m12f_mass56000_cr_700.ski
/nfspool-0/home/jennydw/SKIRT-tools/tutorials/example
/export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt: /lib64/libstdc++.so.6: version `GLIBCXX_3.4.20' not found (required by /export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt)
/export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt: /lib64/libstdc++.so.6: version `CXXABI_1.3.8' not found (required by /export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt)
/export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt: /lib64/libstdc++.so.6: version `CXXABI_1.3.9' not found (required by /export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt)
/export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt: /lib64/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt)
/export/nfs0home/jennydw/SKIRT/release/SKIRT/main/skirt: /lib64/libstdc++.so.6: version `GLIBCXX_3.4.21' not found (required by /e

We can also run the simulation without calling an object, using the following code. To do this, "rundir" should already have all the data and parameter files.

In [ ]:

sktools.run_simulation(name="m12f_f", rundir=rundir, nthreads=4, nprocesses=2)


Lastly, we can run the simulation directly using SKIRT.

To run the simulation, navigate to "rundir" and execute the skirt command.

    cd /nfspool-0/home/jennydw/SKIRT-tools/tutorials/example
    skirt m12f_mass56000_cr_700.ski

You may need to load the gnu module if you encounter an error on greenplanet:

    ml gnu/11.3.0